<a href="https://colab.research.google.com/github/aarthi-2007/GEN-AI-LAB-EXPERIMENTS/blob/main/6th_GenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q sentence-transformers faiss-cpu transformers torch

In [1]:
from sentence_transformers import SentenceTransformer

print("Sentence Transformers imported successfully!")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully!")

Sentence Transformers imported successfully!


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!


In [5]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# --------------------------------------------------
# 1. Knowledge Base
# --------------------------------------------------

documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation combines document retrieval with text generation.",
    "Paris is the capital city of France.",
    "The Eiffel Tower is one of the most famous landmarks in the world.",
    "France is a country located in Western Europe."
]

# --------------------------------------------------
# 2. Load Sentence Transformer
# --------------------------------------------------

print("Loading embedding model...")

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings for all documents
document_embeddings = embedder.encode(
    documents,
    convert_to_numpy=True
)

print("Embeddings created successfully.")

# --------------------------------------------------
# 3. Create FAISS Index
# --------------------------------------------------

dimension = document_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(document_embeddings)

print("FAISS index created successfully.")

# --------------------------------------------------
# 4. User Query
# --------------------------------------------------

query = "Where is the Eiffel Tower located?"

print("\nQuestion:")
print(query)

# --------------------------------------------------
# 5. Convert Query into Embedding
# --------------------------------------------------

query_embedding = embedder.encode(
    [query],
    convert_to_numpy=True
)

# --------------------------------------------------
# 6. Retrieve Relevant Documents
# --------------------------------------------------

k = 2

distances, indices = index.search(
    query_embedding,
    k
)

retrieved_documents = []

for i in indices[0]:
    retrieved_documents.append(documents[i])

# Combine retrieved documents into context
context = " ".join(retrieved_documents)

print("\nRetrieved Context:")
print(context)

# --------------------------------------------------
# 7. Load FLAN-T5 Model
# --------------------------------------------------

print("\nLoading FLAN-T5 model...")

tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-base"
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)

print("FLAN-T5 loaded successfully.")

# --------------------------------------------------
# 8. Create RAG Prompt
# --------------------------------------------------

prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

print("\nPrompt:")
print(prompt)

# --------------------------------------------------
# 9. Tokenize Prompt
# --------------------------------------------------

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# --------------------------------------------------
# 10. Generate Answer
# --------------------------------------------------

with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

# --------------------------------------------------
# 11. Decode Answer
# --------------------------------------------------

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

# --------------------------------------------------
# 12. Display Final Answer
# --------------------------------------------------

print("\n====================================")
print("RAG GENERATED ANSWER")
print("====================================")

print(answer)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings created successfully.
FAISS index created successfully.

Question:
Where is the Eiffel Tower located?

Retrieved Context:
The Eiffel Tower is located in Paris, France and was completed in 1889. The Eiffel Tower is one of the most famous landmarks in the world.

Loading FLAN-T5 model...


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 loaded successfully.

Prompt:

Use the following context to answer the question.

Context:
The Eiffel Tower is located in Paris, France and was completed in 1889. The Eiffel Tower is one of the most famous landmarks in the world.

Question:
Where is the Eiffel Tower located?

Answer:


RAG GENERATED ANSWER
Paris, France
